# importing libraries

In [ ]:
import pandas as pd
import MySQLdb
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)
import json
import urllib.request



# creating engines and connections

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


db = client.get_database('telematics_db')






# setting right pd options

In [ ]:
pd.set_option("display.max_rows",5)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# defining basic functions

In [ ]:
gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x).date()
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)




In [ ]:
def haversine(L1,L2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """

    lon1=L1[0]
    lat1=L1[1]
    lon2=L2[0]
    lat2=L2[1]
    #print(lon1,lat1,lon2,lat2)
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000 # Radius of earth in kilometers. Use 3956 for miles
    return c * r




In [ ]:
haversine([0,0],[22,69])

In [ ]:
def split(start,end,vehicleList1):
    l=[]
    threshhold=86400
    for vid in vehicleList1:
        while(start+threshhold<end):
            l.append([vid,start,start+threshhold])
            start+=threshhold
        l.append([vid,start,end])

    return l

In [ ]:
def analysis(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    global jump_distance
    global total_distace_travelled
    try:   
        
        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
        data=json.load(urllib.request.urlopen(url))
        #print(url)
        df=pd.DataFrame(data['data'][str(vid)])
    
    
    except Exception as e:
        
        print("QUERY ISSUE AAYA HAI  " + str(e))
        df=pd.DataFrame()
        
               
#################################################### CHECKING IF IT NEEDS HEART BEAT OR NOT #########################################################################################                                    

    if ('ignOn' in df)&('chargeOn' in df):
    ############################################################ CHECKING IF THERE IS ANY DATA FOR NO HEART BEAT ##################################################################################################                            
        if len(df)>10:
        
            
        
            print('the number of pings= '+str(len(df)))
            df=df.sort_values('time',ascending=False)

            df.reset_index(drop=True,inplace=True)

            try:
                
                no_info_epochs=[]

                device_history=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':vid,'fromTime':{'$lte':day_end},'toTime':{'$gte':day_start}})))
                #print(len(device_history))

                if len(device_history)>0:
                    device_history.sort_values('fromTime',ascending=True,inplace=True)
                    no_info_epochs=[]
                    for b,brow in device_history.iterrows():
                        try:
                            no_info_start=list(filter(lambda i : i<brow['fromTime'],df.time))[-1] -5
                        except:
                            no_info_start=0
                        try:
                            no_info_end=list(filter(lambda i : i>brow['toTime'],df.time))[0]+5
                        except:
                            no_info_end=999999999999

                        no_info_epochs.append([no_info_start,no_info_end])

                else:
                    no_info_distance_travelled=None
                    #print("NO_NO_INFO")

            except Exception as e:
                device_history=pd.DataFrame()

                no_info_distance_travelled=None
                print('for the no_info_error issue is ' + str(e))



            total_counts=len(df)



            if('usedSat' in df):
                sat_used_values=df['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
                #['mean', '1_pctl', '5_pctl', '10_pctl', '50_pctl']]

            else:
                sat_used_values=[None]*5



            if('gsmPct' in df):
                gsm_pct_values=df['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
                #['mean', '1_pctl', '5_pctl', '10_pctl', '50_pctl']
            else:
                gsm_pct_values=[None]*5



            df.loc[:,'loc2']=df['location'].shift(1)
            df.loc[:,'time1']=df['time'].shift(1)

            df=df[1:]

            time_diff_cons=df['time1']-df['time']

            df.loc[:,'time_diff_cons_pings']=time_diff_cons

            time_diff_cons_pings_values=df.time_diff_cons_pings.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]
            #'mean', '50_pctl', '80_pctl', '95_pctl', '99_pctl'



            df.loc[:,'server_created_time']=df['created']/1000 #.apply(getTime)



            df.loc[:,'server_delay']=df['server_created_time']-df['time']

            server_delay_values=df.server_delay.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]


            loc_list=df[['location','loc2']].values.tolist()

            dis=[]
            for loc in loc_list:
                y=haversine(loc[0],loc[1])
                dis.append(y)

            df['distance']=dis
            
            no_info_distance_travelled=0
            for epoch in no_info_epochs:
                no_info_distance_travelled+=df[df['time']>=epoch[0]][df['time']<=epoch[1]]['distance'].sum()

            jump_values=df[df['distance']>500]['distance'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]


            df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']    
            df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>0.8 else  'stop' ,axis=1)

            driving_count=len(df[df['moving_status']=='running'])

            stop_count=len(df[df['moving_status']=='stop'])


            driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()

            stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()

            day=getDay((df['time'].min()+df['time'].max())/2)

            total_distace_travelled=df['distance'].sum()


            df=df.sort_values('server_created_time',ascending=False)

            df.reset_index(drop=True,inplace=True)


            df['created_time_shift']=df['server_created_time'].shift(1)
            df.at[0,'created_time_shift']=df.iloc[0]['server_created_time']


            df['created_cons_time_diff']=df['created_time_shift']-df['server_created_time']


            live_time=df[df['created_cons_time_diff']<180]['created_cons_time_diff'].sum()
            offline_time=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].sum()

            created_cons_time_diff_values=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]



            if (df.time.min()-day_start)>=180:
                offline_time=offline_time+df.time.min()-day_start
            else:
                live_time=live_time+df.time.min()-day_start


            if (day_end-df.time.max())>=180:
                offline_time=offline_time+day_end-df.time.max()
            else :
                live_time=live_time+day_end-df.time.max()


            jump_distance=df[df['distance']>500]['distance'].sum()
            jump_percent=100*jump_distance/total_distace_travelled

            expected_run_pings_percent=1000*driving_count/driving_time
            expected_stop_pings_percent=3000*stop_count/stop_time


            no_info_duration=df[df['created_cons_time_diff']>3600]['created_cons_time_diff'].sum()

            if (df.time.min()-day_start)>=3600:
                no_info_duration=no_info_duration+df.time.min()-day_start

            elif (day_end-df.time.max())>=3600:
                no_info_duration=no_info_duration+day_end-df.time.max()

            no_info_instances=len(device_history)




            final_values=[vid,day,total_counts,None]+list(sat_used_values)+list(gsm_pct_values)+list(time_diff_cons_pings_values)+list(server_delay_values)+list(jump_values)+[driving_count,stop_count,driving_time,stop_time,total_distace_travelled,live_time,offline_time]+list(created_cons_time_diff_values)+[jump_distance,jump_percent,expected_run_pings_percent,expected_stop_pings_percent,no_info_distance_travelled,no_info_duration,no_info_instances]

            #print(final_values[:-2])

            final_column_names=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                                'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                               'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                               'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                               'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                               'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                               'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                               'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                               'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']


            day_wise=pd.DataFrame([final_values],columns=final_column_names)
            print(final_values)


            return day_wise
        
            
            
            
            
            
    ############################################################ IF THERE IS NO DATA FOR NO HEART BEAT ##################################################################################################                            
            

        else:
            print("DATA NAHI AANE WAALA")
            return pd.DataFrame([[vid,getDay((day_start+day_end)/2),len(df)]+[-1]*47],columns=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                                'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                               'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                               'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                               'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                               'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                               'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                               'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                               'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances'])
        
        
        
     ############################################################ ANALYSIS FOR HEARTBEAT DEVICES ##################################################################################################                            
       

    else:
        print('HEART BEAT')
        if len(df)>10:
            
            
            try:
        
                url=GPS_API_BASE+"/heartbeat/list?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
                hb_data=json.load(urllib.request.urlopen(url))
                #print(url)
                hb=pd.DataFrame(hb_data['data'][str(vid)])

            except Exception as e:

                print("HEARTBEAT ISSUE AAYA HAI  " + str(e))
                hb=pd.DataFrame()
            
#             print('HEART BEAT PINGS ' +str(len(hb)))
                 
        
#             print('the number of pings= '+str(len(df)))
            df=df.sort_values('time',ascending=False)

            df.reset_index(drop=True,inplace=True)

            try:
                no_info_epochs=[]
                device_history=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':vid,'fromTime':{'$lte':day_end},'toTime':{'$gte':day_start}})))
#                 print(len(device_history))

                if len(device_history)>0:
                    device_history.sort_values('fromTime',ascending=True,inplace=True)
                    for b,brow in device_history.iterrows():
                        try:
                            no_info_start=list(filter(lambda i : i<brow['fromTime'],df.time))[-1] -5
                        except:
                            no_info_start=0
                        try:
                            no_info_end=list(filter(lambda i : i>brow['toTime'],df.time))[0]+5
                        except:
                            no_info_end=999999999999

                        no_info_epochs.append([no_info_start,no_info_end])

                else:
                    no_info_distance_travelled=None
#                     print("NO_NO_INFO")

            except Exception as e:
                device_history=pd.DataFrame()
                no_info_distance_travelled=None
                print('for the no_info_error issue is ' + str(e))



            total_counts=len(df)



            if('usedSat' in df):
                sat_used_values=df['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
                #['mean', '1_pctl', '5_pctl', '10_pctl', '50_pctl']]

            else:
                sat_used_values=[None]*5



            if('gsmPct' in hb):
                gsm_pct_values=hb['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
                #['mean', '1_pctl', '5_pctl', '10_pctl', '50_pctl']
            else:
                gsm_pct_values=[None]*5



            df.loc[:,'loc2']=df['location'].shift(1)
            df.loc[:,'time1']=df['time'].shift(1)

            df=df[1:]

            time_diff_cons=df['time1']-df['time']

            df.loc[:,'time_diff_cons_pings']=time_diff_cons

            time_diff_cons_pings_values=df.time_diff_cons_pings.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]
            #'mean', '50_pctl', '80_pctl', '95_pctl', '99_pctl'



            df.loc[:,'server_created_time']=df['created']/1000 #.apply(getTime)



            df.loc[:,'server_delay']=df['server_created_time']-df['time']

            server_delay_values=df.server_delay.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]


            loc_list=df[['location','loc2']].values.tolist()

            dis=[]
            for loc in loc_list:
                y=haversine(loc[0],loc[1])
                dis.append(y)

            df['distance']=dis
            
            no_info_distance_travelled=0
            for epoch in no_info_epochs:
                no_info_distance_travelled+=df[df['time']>=epoch[0]][df['time']<=epoch[1]]['distance'].sum()


            jump_values=df[df['distance']>500]['distance'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]


            df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']    
            df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>0.8 else  'stop' ,axis=1)

            driving_count=len(df[df['moving_status']=='running'])

            stop_count=len(df[df['moving_status']=='stop'])


            driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()

            stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()

            day=getDay((df['time'].min()+df['time'].max())/2)

            total_distace_travelled=df['distance'].sum()


            df=df.sort_values('server_created_time',ascending=False)

            df.reset_index(drop=True,inplace=True)
            hb['created']=hb['created']/1000
            hb=hb.sort_values('created',ascending=False)
            hb.reset_index(drop=True,inplace=True)
            df['created_time_shift']=df['created'].shift(1)
            
            hb.at[0,'created_time_shift']=hb.iloc[0]['created']

            df['created_time_shift']=df['server_created_time'].shift(1)
            df.at[0,'created_time_shift']=df.iloc[0]['server_created_time']
            
            hb['created_cons_time_diff']=hb['created_time_shift']-hb['created']


            df['created_cons_time_diff']=df['created_time_shift']-df['server_created_time']


            live_time=hb[hb['created_cons_time_diff']<180]['created_cons_time_diff'].sum()
            offline_time=hb[hb['created_cons_time_diff']>180]['created_cons_time_diff'].sum()

            created_cons_time_diff_values=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]



            if (hb.created.min()-day_start)>=180:
                offline_time=offline_time+hb.created.min()-day_start
            else:
                live_time=live_time+hb.created.min()-day_start


            if (day_end-hb.created.max())>=180:
                offline_time=offline_time+day_end-hb.created.max()
            else :
                live_time=live_time+day_end-hb.created.max()


            jump_distance=df[df['distance']>500]['distance'].sum()
            jump_percent=100*jump_distance/total_distace_travelled

            expected_run_pings_percent=1000*driving_count/driving_time
            expected_stop_pings_percent=3000*stop_count/stop_time


            no_info_duration=df[df['created_cons_time_diff']>3600]['created_cons_time_diff'].sum()

            if (df.time.min()-day_start)>=3600:
                no_info_duration=no_info_duration+df.time.min()-day_start

            elif (day_end-df.time.max())>=3600:
                no_info_duration=no_info_duration+day_end-df.time.max()
            
            heart_beat_pings=len(hb)
            no_info_instances=len(device_history)






            final_values=[vid,day,total_counts,heart_beat_pings]+list(sat_used_values)+list(gsm_pct_values)+list(time_diff_cons_pings_values)+list(server_delay_values)+list(jump_values)+[driving_count,stop_count,driving_time,stop_time,total_distace_travelled,live_time,offline_time]+list(created_cons_time_diff_values)+[jump_distance,jump_percent,expected_run_pings_percent,expected_stop_pings_percent,no_info_distance_travelled,no_info_duration,no_info_instances]

            #print(final_values[:-2])

            final_column_names=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                                'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                               'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                               'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                               'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                               'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                               'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                               'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                               'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']


            day_wise=pd.DataFrame([final_values],columns=final_column_names)
            print(final_values)


            return day_wise
        
            
             
            
            
    ############################################################ IF THERE IS NO DATA FOR NO HEART BEAT ##################################################################################################                            
            

        else:
            print("DATA NAHI AANE WAALA")
            return pd.DataFrame([[vid,getDay((day_start+day_end)/2),len(df)]+[-1]*47],columns=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                                'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                               'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                               'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                               'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                               'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                               'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                               'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                               'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances'])
        
        

        
        
        
        
        
        


# collecting data 

In [ ]:
aggregate=pd.read_sql("select * from operator_vehicles where deboarding_time is null and vehicle_id=100001 ;",mysql_vds).rename(index=str,columns={"vehicle_id":"vehicleId"})

In [ ]:
#getting old data from metabse 
old=pd.read_sql("select * from device_performance limit 10",con=engine)

In [ ]:
last_date_algo_ran=old.date.max()
start=gettime(last_date_algo_ran)-19800+86400
end=gettime(dt.date.today())-19800+86400


In [ ]:
vehicleList=aggregate.vehicleId.tolist()

vid_start_end=split(start,end,vehicleList)


In [ ]:
print("Total {} iterations to go".format(len(vid_start_end)))
print(len(vehicleList))

In [ ]:
from multiprocessing.dummy import Pool as ThreadPool
pool = ThreadPool(10)
results=pool.map(analysis,vid_start_end)
#results=map(analysis,vid_start_end)
pool.close()
pool.join()

data=pd.concat(results)
        
        
        

In [ ]:
data=data.round(0)

In [ ]:
total_distace_travelled


In [ ]:
vehicle_details=pd.read_sql("select vehicle_id,boarding_time,vehicle_number,data_source,operator_id,new_device,installation_subtype,device_id,is_force_live,operator_name,operator_phone_number,sim_id,model_name,sales_dept,sim_number,sim_phone_number,imei from vehicle_details",con=engine)


In [ ]:
data=pd.merge(data,vehicle_details,on='vehicle_id',how='left')

In [ ]:
devices=pd.read_sql("select created as device_created_time,imei,new_device as device_table_new_device,version as device_version,purchase_date as device_purchase_date from devices",con=mysql_vds)

In [ ]:
data=pd.merge(data,devices, on='imei',how='left')

In [ ]:
 engine.execute(" delete from device_health ")

In [ ]:
data.set_index(['vehicle_number','day','vehicle_id']).to_sql("device_health",con=engine,if_exists='append',chunksize=500)

In [ ]:
datetime.today.strftime("%m/%d/%Y, %H:%M:%S")

In [ ]:
from datetime import datetime,timedelta


In [ ]:
pd.read_sql("select * from device_health where day<{}".format(str("'"+str((datetime.now()-timedelta(days=2)).strftime("%Y-%m-%d"))+str("'"))),con=engine)

In [ ]:
str("'"+str((datetime.now()-timedelta(days=2)).strftime("%Y-%m-%d"))+str("'"))